In [50]:
from typing import Annotated, List, TypedDict, Optional
import operator
from typing import TypedDict, List, Optional, Union
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [51]:
load_dotenv()

True

In [52]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [53]:
class GrammarState(TypedDict):
    text: str
    is_good: bool
    iteration: int

In [54]:
def clean_text(state: GrammarState):
    response = llm.invoke(
        f"Correct only the grammar, spelling, and punctuation in the following text. Preserve the original meaning and do not add any new information. Return only the corrected text:\n{state['text']}"
    )
    return {"text": response.content, "iteration": state["iteration"] + 1}

In [55]:
def evaluate_text(state: GrammarState):
    response = llm.invoke(
        f"Is the following text grammatically correct, clear, and natural? Reply with only 'yes' or 'no'.\n{state['text']}"
    )

    if response.content.strip().lower() == "yes":
        is_good = True
    else:
        is_good = False
    return {"is_good": is_good}

In [56]:
def should_continue(state: GrammarState):
    if state["is_good"] or state["iteration"] >= 3:
        return "end"
    return "continue"

In [57]:
builder = StateGraph(GrammarState)

builder.add_node("clean", clean_text)
builder.add_node("evaluate", evaluate_text)

builder.add_edge(START, "clean")

builder.add_edge("clean", "evaluate")

builder.add_conditional_edges(
    "evaluate", should_continue, {"continue": "clean", "end": END}
)

app = builder.compile()

In [59]:
result = app.invoke({"text": "My names is Rohan", "iteration": 0, "is_good": False})

print(result)

{'text': 'My name is Rohan.', 'is_good': True, 'iteration': 1}
